In [7]:
"""
Generador de demanda vehicular estocástica (Poisson) para SUMO
"""
import numpy as np
import xml.etree.ElementTree as ET
from xml.dom import minidom

In [ ]:
# 1. PARÁMETROS GENERALES
SEED = 42
DURACION = 3600
INTERVALO = 60

In [ ]:
# 2. DEFINICIONES
ACCESOS = [
    {"nombre": "Oeste", "rutas": [("Oeste_Este", "-E4", "E3", 0.80), ("Oeste_Sur", "-E4", "E6", 0.20)]},
    {"nombre": "Norte", "rutas": [("Norte_Sur", "-E5", "E6", 0.85), ("Norte_Este", "-E5", "E3", 0.15)]}
]

TIPOS = [
    ("liviano", 0.85),
    ("bus", 0.10),
    ("pesado", 0.05)
]

In [10]:
# 3. LÓGICA DE GENERACIÓN (POISSON)
def generar_tiempos_acceso(total_esperado, seed_offset):
    rng = np.random.default_rng(SEED + seed_offset)
    tiempos = []
    tasa_constante = total_esperado * (INTERVALO / DURACION)
    for t0 in range(0, DURACION, INTERVALO):
        n = rng.poisson(tasa_constante)
        offsets = rng.uniform(0, INTERVALO, size=n)
        tiempos.extend(t0 + offsets)
    tiempos.sort()
    return tiempos

In [11]:
# 4. CONSTRUCCIÓN XML
def construir_xml(vehiculos_por_acceso):
    root = ET.Element("routes")
    # Definición de vTypes
    ET.SubElement(root, "vType", id="liviano", accel="2.6", decel="4.5", length="4.5", maxSpeed="14.0")
    ET.SubElement(root, "vType", id="bus", accel="1.2", decel="4.0", length="10.0", maxSpeed="11.0", vClass="bus")
    ET.SubElement(root, "vType", id="pesado", accel="1.0", decel="3.5", length="12.0", maxSpeed="10.0", vClass="truck")

    for acceso in ACCESOS:
        for rid, e_from, e_to, _ in acceso["rutas"]:
            ET.SubElement(root, "route", id=rid, edges=f"{e_from} {e_to}")

    todos = []
    for acceso, (tiempos, seed) in zip(ACCESOS, vehiculos_por_acceso):
        rng = np.random.default_rng(seed)
        rutas = acceso["rutas"]
        p_r = np.array([r[3] for r in rutas])
        p_t = np.array([t[1] for t in TIPOS])
        
        for t in tiempos:
            ruta_info = rng.choice(rutas, p=p_r/p_r.sum())
            tipo = rng.choice([t[0] for t in TIPOS], p=p_t/p_t.sum())
            todos.append({"depart": t, "type": tipo, "route": ruta_info[0]})

    todos.sort(key=lambda x: x["depart"])
    for i, v in enumerate(todos):
        ET.SubElement(root, "vehicle", id=f"veh_{i}", type=v["type"], route=v["route"], depart=f"{v['depart']:.2f}")
    return root, len(todos)

def guardar(root, filepath):
    xml_str = ET.tostring(root, encoding='utf-8')
    parsed_str = minidom.parseString(xml_str)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(parsed_str.toprettyxml(indent="    "))

In [12]:
# 5. EJECUCIÓN
if __name__ == "__main__":
    tiempos_oeste = generar_tiempos_acceso(1205, 1)
    tiempos_norte = generar_tiempos_acceso(1205, 11)
    vehiculos_por_acceso = [(tiempos_oeste, SEED+1), (tiempos_norte, SEED+11)]
    root, total = construir_xml(vehiculos_por_acceso)
    guardar(root, "loja_intersection_poisson.rou.xml")
    print(f"✓ Archivo creado con {total} vehículos.")

✓ Archivo creado con 2456 vehículos.
